# 📊 Student Productivity Analysis
**Extended Python Analysis — Pandas · Scikit-learn · Seaborn**

This notebook extends the SQL-based analysis (PostgreSQL + Metabase) with:
- Data cleaning & exploratory data analysis using **Pandas**
- Interactive visualizations using **Matplotlib & Seaborn**
- Behavioral segmentation using **K-Means Clustering**
- Productivity prediction using **Linear Regression**

**Dataset:** [Student Productivity and Behavior Dataset (20K)](https://www.kaggle.com/datasets/algozee/student-productivity-and-behavior-dataset-20k)  
**Author:** Muhammad Arya Rumanga A. Baso  
**GitHub:** [github.com/rumanga09](https://github.com/rumanga09)

---
## 0. Setup & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully!')

---
## 1. Load Data

In [ ]:
df = pd.read_csv('student_productivity_distraction_dataset_20000.csv')

print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

---
## 2. Data Cleaning & EDA

In [ ]:
print('=== Data Quality Check ===')
print(f'Null values  : {df.isnull().sum().sum()}')
print(f'Duplicates   : {df.duplicated().sum()}')
print(f'Shape        : {df.shape}')
print()
df.describe().round(2)

In [ ]:
# Feature engineering
df['total_distraction'] = (
    df['phone_usage_hours']
    + df['social_media_hours']
    + df['youtube_hours']
    + df['gaming_hours']
)

# Productivity segmentation (mirrors SQL segmentation)
def segment(score):
    if score < 40:   return 'Low'
    elif score < 70: return 'Medium'
    else:            return 'High'

df['segment'] = df['productivity_score'].apply(segment)

print('Segment Distribution:')
print(df['segment'].value_counts())
print()
print('Avg Metrics by Segment:')
df.groupby('segment')[[
    'sleep_hours', 'study_hours_per_day', 'total_distraction',
    'stress_level', 'exercise_minutes', 'attendance_percentage', 'productivity_score'
]].mean().round(2)

In [ ]:
print('Gender Distribution:')
print(df['gender'].value_counts())
print()
print('Avg Productivity by Gender:')
df.groupby('gender')['productivity_score'].mean().round(2)

---
## 3. Visualizations

In [ ]:
seg_order  = ['Low', 'Medium', 'High']
seg_colors = {'Low': '#E74C3C', 'Medium': '#F39C12', 'High': '#2ECC71'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Student Productivity — Exploratory Analysis', fontsize=16, fontweight='bold')

# 3a. Productivity score distribution
for seg in seg_order:
    data = df[df['segment'] == seg]['productivity_score']
    axes[0, 0].hist(data, bins=30, alpha=0.6, label=seg, color=seg_colors[seg])
axes[0, 0].set_title('Productivity Score Distribution by Segment')
axes[0, 0].set_xlabel('Productivity Score')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()

# 3b. Sleep vs Productivity
sns.scatterplot(data=df.sample(1000, random_state=42), x='sleep_hours', y='productivity_score',
                hue='segment', hue_order=seg_order, palette=seg_colors, alpha=0.6, ax=axes[0, 1])
axes[0, 1].set_title('Sleep Hours vs Productivity Score')
axes[0, 1].set_xlabel('Sleep Hours')
axes[0, 1].set_ylabel('Productivity Score')

# 3c. Avg total distraction by segment
distraction_avg = df.groupby('segment')['total_distraction'].mean().reindex(seg_order)
bars = axes[1, 0].bar(seg_order, distraction_avg,
                       color=[seg_colors[s] for s in seg_order], edgecolor='white')
for bar, val in zip(bars, distraction_avg):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                    f'{val:.1f}h', ha='center', fontsize=10, fontweight='bold')
axes[1, 0].set_title('Avg Total Distraction by Segment')
axes[1, 0].set_xlabel('Segment')
axes[1, 0].set_ylabel('Hours/day')

# 3d. Correlation heatmap
corr_cols = ['sleep_hours', 'study_hours_per_day', 'phone_usage_hours',
             'stress_level', 'exercise_minutes', 'attendance_percentage',
             'focus_score', 'productivity_score']
corr = df[corr_cols].corr().round(2)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            ax=axes[1, 1], linewidths=0.5)
axes[1, 1].set_title('Correlation Heatmap')

plt.tight_layout()
plt.savefig('productivity_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Lifestyle factors
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Lifestyle Factors vs Productivity', fontsize=14, fontweight='bold')

sns.scatterplot(data=df.sample(1000, random_state=42), x='exercise_minutes', y='productivity_score',
                hue='segment', hue_order=seg_order, palette=seg_colors, alpha=0.6, ax=axes[0])
axes[0].set_title('Exercise Minutes vs Productivity')
axes[0].set_xlabel('Exercise (minutes/day)')
axes[0].set_ylabel('Productivity Score')

sns.scatterplot(data=df.sample(1000, random_state=42), x='attendance_percentage', y='productivity_score',
                hue='segment', hue_order=seg_order, palette=seg_colors, alpha=0.6, ax=axes[1])
axes[1].set_title('Attendance % vs Productivity')
axes[1].set_xlabel('Attendance Percentage')
axes[1].set_ylabel('Productivity Score')

plt.tight_layout()
plt.show()

---
## 4. K-Means Clustering

In [ ]:
features_cluster = [
    'sleep_hours', 'study_hours_per_day', 'total_distraction',
    'stress_level', 'exercise_minutes', 'attendance_percentage'
]
X_cluster = df[features_cluster].copy()
scaler    = StandardScaler()
X_scaled  = scaler.fit_transform(X_cluster)

inertias = []
k_range  = range(2, 8)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(k_range), inertias, 'o-', color='#2C3E50', linewidth=2, markersize=7)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.6, label='k=3 selected')
plt.title('Elbow Method — Finding Optimal K')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
kmeans        = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

cluster_means  = df.groupby('cluster')['productivity_score'].mean().sort_values()
cluster_labels = {idx: label for idx, label in
                  zip(cluster_means.index, ['Low Performer', 'Mid Performer', 'High Performer'])}
df['cluster_label'] = df['cluster'].map(cluster_labels)

cluster_palette = {'Low Performer': '#E74C3C', 'Mid Performer': '#F39C12', 'High Performer': '#2ECC71'}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=df.sample(1500, random_state=42), x='sleep_hours', y='productivity_score',
                hue='cluster_label', palette=cluster_palette, alpha=0.6, ax=axes[0])
axes[0].set_title('Clusters: Sleep vs Productivity')

sns.scatterplot(data=df.sample(1500, random_state=42), x='total_distraction', y='productivity_score',
                hue='cluster_label', palette=cluster_palette, alpha=0.6, ax=axes[1])
axes[1].set_title('Clusters: Distraction vs Productivity')

plt.tight_layout()
plt.savefig('clustering_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('Cluster Profiles:')
df.groupby('cluster_label')[features_cluster + ['productivity_score']].mean().round(2)

---
## 5. Linear Regression — Predict Productivity Score

In [ ]:
features_reg = [
    'sleep_hours', 'study_hours_per_day', 'phone_usage_hours',
    'social_media_hours', 'stress_level', 'exercise_minutes',
    'attendance_percentage', 'focus_score', 'total_distraction'
]
X = df[features_reg]
y = df['productivity_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f'R2 Score : {r2:.4f}  ->  {r2*100:.1f}% variance explained')
print(f'MAE      : {mae:.2f} points')
print()

coef_df = pd.DataFrame({
    'Feature': features_reg,
    'Coefficient': model.coef_.round(3)
}).sort_values('Coefficient', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors_coef = ['#2ECC71' if c > 0 else '#E74C3C' for c in coef_df['Coefficient']]
axes[0].barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_coef, edgecolor='white')
axes[0].axvline(x=0, color='black', linewidth=0.8)
axes[0].set_title('Feature Coefficients\n(Green = positive, Red = negative)')

axes[1].scatter(y_test, y_pred, alpha=0.3, color='#3498DB', s=10)
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=1.5)
axes[1].set_title(f'Actual vs Predicted  |  R2 = {r2:.3f}')
axes[1].set_xlabel('Actual Score')
axes[1].set_ylabel('Predicted Score')

plt.tight_layout()
plt.savefig('regression_results.png', dpi=150, bbox_inches='tight')
plt.show()

coef_df

---
## 6. Key Insights & Conclusions

In [ ]:
high = df[df['segment'] == 'High']
low  = df[df['segment'] == 'Low']

print('=' * 55)
print('KEY INSIGHTS')
print('=' * 55)
print(f'''
1. Sleep & Productivity
   High performers sleep avg {high["sleep_hours"].mean():.1f}h
   vs {low["sleep_hours"].mean():.1f}h for Low performers

2. Digital Distraction Gap
   High performers: {high["total_distraction"].mean():.1f}h/day
   Low performers : {low["total_distraction"].mean():.1f}h/day
   Difference     : {abs(low["total_distraction"].mean() - high["total_distraction"].mean()):.1f}h per day

3. Exercise & Attendance
   High performers exercise avg {high["exercise_minutes"].mean():.0f} min/day
   vs {low["exercise_minutes"].mean():.0f} min/day for Low performers
   High attendance avg: {high["attendance_percentage"].mean():.1f}%
   Low attendance avg : {low["attendance_percentage"].mean():.1f}%

4. Stress Levels
   High performers avg stress: {high["stress_level"].mean():.1f}/10
   Low performers avg stress : {low["stress_level"].mean():.1f}/10

5. Predictive Model
   Linear Regression R2 = {r2:.2f}
   Strongest positive predictors: study hours, attendance, focus score
   Strongest negative predictors: phone usage, stress

6. Clustering
   K-Means (k=3) identified 3 distinct behavioral profiles
   aligning with Low / Mid / High productivity segments
''')

---
## Future Improvements

- [ ] Add Random Forest or XGBoost for better predictive accuracy
- [ ] Perform SHAP analysis for model explainability
- [ ] Build an interactive dashboard with Streamlit
- [ ] Conduct deeper gender-based analysis
- [ ] Deploy public dashboard instance